In [2]:
import torch
from torch.nn import functional as F
torch.manual_seed(42)
B,T,C = 4,8,32 # batch time channels
x = torch.randn(B,T,C)

head_size = 16

key = torch.nn.Linear(C,head_size,bias=False)
query = torch.nn.Linear(C,head_size,bias=False)
value = torch.nn.Linear(C,head_size,bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2,-1) * head_size**-0.5

tril = torch.tril(torch.ones(T,T))

wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=1)
v = value(x)
out = wei@v
out.shape

torch.Size([4, 8, 16])

In [3]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1)* head_size**-.5

In [ ]:
k.var(),q.var(),wei.var()

In [3]:
class BatchNorm1d:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.ones(dim)
    def __call__(self,x):
        xmean = x.mean(1,keepdim=True)
        xvar = x.var(1,keepdim=True)
        xhat = (x-xmean)/torch.sqrt(xvar +self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(3524)
module = BatchNorm1d(100)

x = torch.randn(32,100)

x = module(x)
x.shape

torch.Size([32, 100])